# Scenario API Smoke Test

This notebook tests the basic functionality of the scenario API.

## A. Importit

In [2]:
import sys
from pathlib import Path

# Locate the repository root by searching for extensions/scenario_api
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'extensions' / 'scenario_api').exists()), None)
if repo_root is None:
    raise RuntimeError('Could not locate extensions/scenario_api from current working directory')

extensions_dir = repo_root / 'extensions'
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import *


ModuleNotFoundError: No module named 'scenario_api'

## B. Parametrilohkot

In [ ]:
# Variant block
variant_block = create_block(
    name="variant",
    params={"relative_transmission": 1.3, "asymptomatic_fraction": 0.35}
)

# Testing block
testing_block = create_block(
    name="testing",
    params={"testing_rate": 0.2}
)

print("Blocks created:")
print(variant_block)
print(testing_block)

## C. Verkkomääritykset

In [ ]:
# Network specs (updated for validated kinds)
household_spec = create_network_spec(
    name="households",
    kind="household",
    config={"population_size": 1000}
)

work_spec = create_network_spec(
    name="work",
    kind="activity_structured",
    config={"mean_contacts": 10, "activation_prob": 0.5}
)

community_spec = create_network_spec(
    name="community",
    kind="activity_random",
    config={"mean_contacts": 4, "dispersion": 2}
)

print("Network specs created:")
print(household_spec)
print(work_spec)
print(community_spec)


## D. Aikajanan tapahtumat

In [ ]:
# Event at step 10: set relative_transmission to 0.9
event1 = create_event(
    time=10,
    action="set",
    target="relative_transmission",
    value=0.9
)

# Event at step 20: scale testing_rate by 1.5
event2 = create_event(
    time=20,
    action="scale",
    target="testing_rate",
    value=1.5
)

print("Events created:")
print(event1)
print(event2)

## E. Skenaario

In [ ]:
# Create scenario
scenario = create_scenario(
    name="test_scenario",
    base_params={"population_size": 10000, "relative_transmission": 1.0, "testing_rate": 0.1},
    blocks=[variant_block, testing_block],
    network_specs=[household_spec, community_spec],
    events=[event1, event2]
)

print("Scenario created:")
print(scenario)

# Resolve scenario
resolved = resolve_scenario(scenario)
print("\nResolved scenario:")
print(resolved)

## F. Dummy-runner

In [ ]:
# Run the scenario
result = run_scenario(resolved, steps=30)
print("Simulation result:")
print(result)

## G. Tulos aikasarjaksi

In [ ]:
# Convert to time series
ts = result_to_timeseries(result, "cases")
print("Time series:")
print(ts)

# Simple plot
import matplotlib.pyplot as plt
plt.plot(ts.times, ts.values)
plt.xlabel('Time')
plt.ylabel('Cases')
plt.title('Simulated Cases')
plt.show()

## H. Havaintodata

In [ ]:
# Create observed data
observed_data = {
    "time": list(range(30)),
    "cases": [i * 0.5 + 1 for i in range(30)]
}

dataset = load_observed_dataset("test_data", observed_data)
obs_ts = dataset_to_timeseries(dataset, "cases")
print("Observed time series:")
print(obs_ts)

## I. Vertailu

In [ ]:
# Plot both
plt.plot(ts.times, ts.values, label='Simulated')
plt.plot(obs_ts.times, obs_ts.values, label='Observed')
plt.xlabel('Time')
plt.ylabel('Cases')
plt.title('Comparison')
plt.legend()
plt.show()